# 01 – Explore Complaints

Understand the structure of NHTSA's vehicle complaints file before building anything.

**Data:** `COMPLAINTS_RECEIVED_2025-2026.txt` from [NHTSA Datasets and APIs](https://www.nhtsa.gov/nhtsa-datasets-and-apis)

**Data dictionary:** [CMPL.txt](https://static.nhtsa.gov/odi/ffdd/cmpl/CMPL.txt)

In [1]:
import csv
import pandas as pd

pd.set_option("display.max_colwidth", 200)

## Load the data

Per CMPL.txt, the file is tab-separated with no header row. Narratives contain quote marks, so quoting is turned off, and the file is UTF-8 (reading it as latin-1 garbles apostrophes into `â`).

In [2]:
COMPLAINTS_PATH = "../data/raw/COMPLAINTS_RECEIVED_2025-2026.txt"

# Field order from CMPL.txt (51 fields as of the April 30, 2026 update)
CMPL_COLUMNS = [
    "cmplid", "odino", "mfr_name", "maketxt", "modeltxt", "yeartxt", "crash",
    "faildate", "fire", "injured", "deaths", "compdesc", "city", "state", "vin",
    "datea", "ldate", "miles", "occurences", "cdescr", "cmpl_type",
    "police_rpt_yn", "purch_dt", "orig_owner_yn", "anti_brakes_yn",
    "cruise_cont_yn", "num_cyls", "drive_train", "fuel_sys", "fuel_type",
    "trans_type", "veh_speed", "dot", "tire_size", "loc_of_tire",
    "tire_fail_type", "orig_equip_yn", "manuf_dt", "seat_type",
    "restraint_type", "dealer_name", "dealer_tel", "dealer_city",
    "dealer_state", "dealer_zip", "prod_type", "repaired_yn", "medical_attn",
    "vehicles_towed_yn", "state_of_incident", "vehicle_operator",
]
len(CMPL_COLUMNS)

51

In [3]:
complaints = pd.read_csv(
    COMPLAINTS_PATH,
    sep="\t",
    header=None,
    names=CMPL_COLUMNS,
    encoding="utf-8",
    encoding_errors="replace",
    quoting=csv.QUOTE_NONE,
)
complaints.shape

/var/folders/y_/75nvqn5s78748v1dk_k8x5480000gn/T/ipykernel_13780/178582350.py:1: DtypeWarning: Columns (29: fuel_type, 35: tire_fail_type, 41: dealer_tel, 44: dealer_zip, 49: state_of_incident, 50: vehicle_operator) have mixed types. Specify dtype option on import or set low_memory=False.
  complaints = pd.read_csv(


(197171, 51)

In [4]:
complaints[["maketxt", "modeltxt", "yeartxt", "compdesc", "ldate", "cdescr"]].head(10)

,maketxt,modeltxt,yeartxt,compdesc,ldate,cdescr
0,FORD,EXPLORER,2016,STRUCTURE:BODY,20250101,A-pillar trim flew off while driving. Obviously could've caused harm to other drivers. Ford has not notified me of when my vehicle will be repaired. Dealership offered to repair at cost to me.
1,JEEP,GLADIATOR,2022,STEERING,20250101,"Occasionally, while driving on normal roads, my steering wheel locks up completely when I am changing lanes or accelerating. This issue typically resolves only when I reduce my speed to around 20-..."
2,TOYOTA,TUNDRA,2024,SERVICE BRAKES,20250101,"Brakes squeaked since date pic purchased. We brought it up to the dealer’s attention at Longo Toyota in El Monte,CA during a service. They mentioned they know about it as serval tundra owners have..."
3,BUICK,ENVISTA,2024,ELECTRICAL SYSTEM,20250101,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...
4,BUICK,ENVISTA,2024,VEHICLE SPEED CONTROL,20250101,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...
5,BUICK,ENVISTA,2024,UNKNOWN OR OTHER,20250101,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...
6,HONDA,PROLOGUE,2024,STEERING,20250101,Vehicle Drive Shaft makes clicking noise at the CV Joint. Vehicle part is backordered due to nationwide shortage caused by vehicle defect on numerous 2024 Honda Prologues. There is no code on dash...
7,HONDA,PROLOGUE,2024,SUSPENSION,20250101,Vehicle Drive Shaft makes clicking noise at the CV Joint. Vehicle part is backordered due to nationwide shortage caused by vehicle defect on numerous 2024 Honda Prologues. There is no code on dash...
8,TESLA,MODEL Y,2023,BACK OVER PREVENTION: SENSING SYSTEM: CAMERA,20250101,"Cameras not working suddenly, navigation software hung in last location coordinates, auto pilot and it’s safety features not working Backup cameras not working , creating blind spot while revers..."
9,VOLKSWAGEN,ATLAS,2018,ENGINE,20250101,A dangerous cam shaft failure led to sensors near the cam shaft and a piece of the metal housing being broken off due to the cam shaft failure. The cam shaft failed at normal highway speeds and st...


## Dates

`ldate` (date NHTSA received the complaint) loads as an integer like `20250101`, so date math would be wrong. Convert it to a real date.

In [5]:
complaints["ldate"] = pd.to_datetime(complaints["ldate"], format="%Y%m%d")
complaints["ldate"].dtype

dtype('<M8[us]')

## Filter to vehicles

`prod_type`: V = vehicle, T = tires, C = child restraint, E = equipment. Only vehicles are in scope for this project.

In [6]:
complaints["prod_type"].value_counts()

prod_type
V    194420
T      1316
C       730
E       705
Name: count, dtype: int64

In [7]:
vehicles = complaints[complaints["prod_type"] == "V"]

# Check: rows removed should equal the T + C + E counts above
print(len(vehicles))
print(len(complaints) - len(vehicles))

194420
2751


## Components

`compdesc` is hierarchical, with `:` separating levels (e.g. `FORWARD COLLISION AVOIDANCE: AUTOMATIC EMERGENCY BRAKING`).

In [8]:
vehicles["compdesc"].value_counts().head(20)

compdesc
ENGINE                                                      31566
ELECTRICAL SYSTEM                                           25455
POWER TRAIN                                                 22443
UNKNOWN OR OTHER                                            19427
SERVICE BRAKES                                               9910
STEERING                                                     9898
FUEL/PROPULSION SYSTEM                                       9527
AIR BAGS                                                     5396
VEHICLE SPEED CONTROL                                        4418
FORWARD COLLISION AVOIDANCE: AUTOMATIC EMERGENCY BRAKING     4321
ENGINE AND ENGINE COOLING                                    4178
STRUCTURE:BODY                                               3995
SUSPENSION                                                   3821
FORWARD COLLISION AVOIDANCE: WARNINGS                        3226
EXTERIOR LIGHTING                                            3009
V

Some values contain commas. Checked whether commas separate multiple components: they don't. Commas are part of single component names (e.g. `FUEL SYSTEM, GASOLINE`), so `compdesc` should never be split on commas.

In [9]:
has_comma = vehicles["compdesc"].str.contains(",")
print(has_comma.sum())
vehicles.loc[has_comma, "compdesc"].value_counts().head(20)

4752


compdesc
SERVICE BRAKES, HYDRAULIC                                                                        2136
FUEL SYSTEM, GASOLINE                                                                            1597
FUEL SYSTEM, DIESEL                                                                               293
VISIBILITY:GLASS, SIDE/REAR                                                                        86
Chest Clip, Buckle, Harness                                                                        81
FUEL SYSTEM, GASOLINE:DELIVERY:FUEL PUMP                                                           50
Carry Handle, Shell, Base                                                                          48
Insert, Padding                                                                                    31
FUEL SYSTEM, GASOLINE:FUEL INJECTION SYSTEM:INJECTORS                                              29
FUEL SYSTEM, GASOLINE:STORAGE:EVAPORATIVE EMISSIONS:CANISTER             

## One row = one complaint-component pair

`odino` is NHTSA's complaint number. A complaint about several components is stored as several rows with the same `odino`, so counting rows overcounts complaints.

In [10]:
vehicles["odino"].duplicated().sum()

np.int64(62786)

In [11]:
dup_id = vehicles.loc[vehicles["odino"].duplicated(), "odino"].iloc[0]
vehicles.loc[vehicles["odino"] == dup_id, ["odino", "cmplid", "maketxt", "modeltxt", "compdesc", "cdescr"]]

,odino,cmplid,maketxt,modeltxt,compdesc,cdescr
3,11633475,2052308,BUICK,ENVISTA,ELECTRICAL SYSTEM,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...
4,11633475,2052309,BUICK,ENVISTA,VEHICLE SPEED CONTROL,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...
5,11633475,2052310,BUICK,ENVISTA,UNKNOWN OR OTHER,While driving down the road to work on the expressway both my screens blacked out couldn’t see my speed and then came back on it did this multiples times on my 30 minute drive to work one way and ...


In [12]:
# Number of distinct complaints
vehicles["odino"].nunique()

131634

## Narratives

In [13]:
vehicles[["cdescr"]].head(3)

,cdescr
0,A-pillar trim flew off while driving. Obviously could've caused harm to other drivers. Ford has not notified me of when my vehicle will be repaired. Dealership offered to repair at cost to me.
1,"Occasionally, while driving on normal roads, my steering wheel locks up completely when I am changing lanes or accelerating. This issue typically resolves only when I reduce my speed to around 20-..."
2,"Brakes squeaked since date pic purchased. We brought it up to the dealer’s attention at Longo Toyota in El Monte,CA during a service. They mentioned they know about it as serval tundra owners have..."


## Findings

- File is tab-separated, no header, UTF-8, with quote marks inside narratives.
- `ldate` must be converted from an integer to a datetime.
- About 98.6% of rows are vehicle complaints (`prod_type == "V"`); the rest are tires, child seats, and equipment.
- Components are hierarchical (`:`), and commas appear inside single component names, so never split on commas.
- One complaint can span several rows (one per component): count unique `odino` for complaint totals.
- Some narratives mention existing recalls, so complaints filed after a recall may partly be reactions to it. Matters for labels later.

*Counts above are from the file as downloaded; NHTSA updates it daily, so exact numbers will drift.*